# Módulo 2 · Amazon S3 con Python (boto3)

Réplica en Python de las prácticas de las diapositivas **18** (*Crear el bucket y subir archivos*) y **19** (*Subir ventas particionadas con Python*).

| # | Ejercicio | Equivalente en la CLI |
|---|---|---|
| 0 | Verificar identidad | `aws sts get-caller-identity` |
| 1 | Crear el bucket | `aws s3 mb s3://$BUCKET --region us-east-2` |
| 2 | Revisar el bloqueo de acceso público | `aws s3api get-public-access-block` |
| 3 | Subir un archivo puntual a `raw/` | `aws s3 cp` |
| 4 | Sincronizar una carpeta | `aws s3 sync` |
| 5 | Listar con tamaños legibles | `aws s3 ls --recursive --human-readable` |
| 6 | Subir ventas en Parquet particionadas `year=/month=/day=` | script de la diapositiva 19 |
| 7 | Leer de vuelta desde S3 | — |
| 8 | Limpieza (opcional) | `aws s3 rb --force` |

**Requisitos:** usuario IAM con `AmazonS3FullAccess` y su access key (`.csv`) en `fundamentos-aws/credentials/`, o un perfil `curso` creado con `aws configure`.

## Configuración

Cambia `BUCKET` por tu alias: el nombre debe ser único en todo AWS (3–63 caracteres, minúsculas, números, puntos y guiones).

### Los datos de partida

Partimos de datos que **ya produjo un pipeline ETL** y que están en `fundamentos-aws/data/`:

| Carpeta | Contenido | Zona del data lake |
|---|---|---|
| `landing/<fuente>/<fecha>/` | Extracciones crudas de cada fuente: `erp/` y `web/` (`pedidos.json`), `crm/` (`clientes.csv`) | raw |
| `staging/<fecha>/` | `ventas.csv`, `clientes.csv` y `productos.csv` ya limpios: sin duplicados ni cantidades inválidas | processed |
| `reportes/` | Un resumen por día (`reporte_<fecha>.csv`) | curated |

Son 8 días, del 20 al 27 de octubre de 2026. En este notebook los subimos a S3.

In [16]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import io
import tempfile

import boto3
import pandas as pd
from botocore.exceptions import ClientError, ProfileNotFound

BUCKET = "de-tecsup-marcell"   # <- reemplaza con tu alias
REGION = "us-east-2"           # tu cuenta (plan gratuito) solo permite us-east-2: una SCP bloquea us-east-1
PERFIL = "curso"

# Raíz del proyecto fundamentos-aws: la primera carpeta hacia arriba que contenga pyproject.toml
PROYECTO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
DATA = PROYECTO / "data"
STAGING = DATA / "staging"        # ventas limpias por día: staging/<fecha>/ventas.csv
REPORTES = DATA / "reportes"      # agregados diarios: reportes/reporte_<fecha>.csv
CREDENCIALES = PROYECTO / "credentials"

print("Proyecto:", PROYECTO)
print("Días en staging:", sorted(p.name for p in STAGING.iterdir() if p.is_dir()))

Proyecto: /Users/marcell/Documents/codigo_by_tecsup/dataengineer-g1/fundamentos-aws
Días en staging: ['2026-10-20', '2026-10-21', '2026-10-22', '2026-10-23', '2026-10-24', '2026-10-25', '2026-10-26', '2026-10-27']


## Credenciales

boto3 recorre la misma cadena de proveedores que la CLI (diapositiva 12): parámetros → variables de entorno → `~/.aws/` → rol del recurso.

La celda intenta primero el perfil `curso`. Si no existe, lee la access key del `.csv` descargado de IAM.

> ⚠️ Nunca imprimas ni subas a Git la clave secreta. La carpeta `credentials/` está en `.gitignore`.

In [17]:
def leer_csv_credenciales(carpeta: Path) -> dict:
    """Lee el .csv que descarga IAM (columnas 'Access key ID' y 'Secret access key')."""
    archivo = next(carpeta.glob("*accessKeys*.csv"))
    with archivo.open(encoding="utf-8-sig") as f:   # utf-8-sig quita el BOM que agrega AWS
        fila = next(csv.DictReader(f))
    return {
        "aws_access_key_id": fila["Access key ID"],
        "aws_secret_access_key": fila["Secret access key"],
    }


try:
    session = boto3.Session(profile_name=PERFIL, region_name=REGION)
    origen = f"perfil '{PERFIL}' de ~/.aws/"
except ProfileNotFound:
    session = boto3.Session(**leer_csv_credenciales(CREDENCIALES), region_name=REGION)
    origen = "archivo .csv de credentials/"

print("Credenciales tomadas de:", origen)

Credenciales tomadas de: archivo .csv de credentials/


## Ejercicio 0 · ¿Con quién estoy trabajando?

El "hola mundo" de AWS. Debe mostrar tu usuario IAM, no el root.

In [18]:
identidad = session.client("sts").get_caller_identity()
print("ARN:   ", identidad["Arn"])
print("Cuenta:", identidad["Account"])

s3 = session.client("s3")

ARN:    arn:aws:iam::073813807844:user/marcell-de
Cuenta: 073813807844


## Ejercicio 1 · Crear el bucket

Equivale a `aws s3 mb s3://$BUCKET --region us-east-2`.

Detalle de la API: en **us-east-1** no se envía `CreateBucketConfiguration`; en cualquier otra región es obligatorio indicar `LocationConstraint`.

In [19]:
def crear_bucket(nombre: str, region: str) -> None:
    params = {"Bucket": nombre}
    if region != "us-east-1":
        params["CreateBucketConfiguration"] = {"LocationConstraint": region}
    try:
        s3.create_bucket(**params)
        print(f"Bucket creado: s3://{nombre} ({region})")
    except ClientError as e:
        codigo = e.response["Error"]["Code"]
        if codigo == "BucketAlreadyOwnedByYou":
            print(f"El bucket s3://{nombre} ya es tuyo: continuamos.")
        elif codigo == "BucketAlreadyExists":
            raise SystemExit(f"'{nombre}' ya lo usa otra cuenta de AWS. Cambia BUCKET y vuelve a ejecutar.")
        else:
            raise
    s3.get_waiter("bucket_exists").wait(Bucket=nombre)


crear_bucket(BUCKET, REGION)
print("Región del bucket:", s3.get_bucket_location(Bucket=BUCKET)["LocationConstraint"] or "us-east-1")

Bucket creado: s3://de-tecsup-marcell (us-east-2)
Región del bucket: us-east-2


## Ejercicio 2 · Confirmar que el acceso público está bloqueado

Equivale a `aws s3api get-public-access-block --bucket $BUCKET`. En buckets nuevos las 4 opciones vienen en `True`: déjalas así.

Un bucket puede volverse público por **dos vías**: las **ACL** (permisos antiguos por objeto o bucket) y la **bucket policy** (documento JSON como el de la diapositiva 9). Para cada vía hay dos opciones: una **impide crear** permisos públicos nuevos y otra **anula** los que ya existen.

| Opción | Vía | Qué hace | Ejemplo de lo que bloquea |
|---|---|---|---|
| `BlockPublicAcls` | ACL | **Prevención.** Rechaza cualquier petición que intente poner una ACL pública en el bucket o en un objeto (`PutBucketAcl`, `PutObjectAcl`, o `PutObject` con ACL pública). | `s3.upload_file(..., ExtraArgs={"ACL": "public-read"})` falla con `AccessDenied`. |
| `IgnorePublicAcls` | ACL | **Neutraliza lo existente.** S3 ignora las ACL públicas que ya tengan el bucket o sus objetos: siguen escritas, pero no dan acceso. | Un objeto subido hace años con `public-read` deja de poder leerse desde internet. |
| `BlockPublicPolicy` | Policy | **Prevención.** Rechaza `PutBucketPolicy` si la política concede acceso público, por ejemplo `"Principal": "*"` sin condiciones. | No puedes guardar una política que permita `s3:GetObject` a cualquiera. |
| `RestrictPublicBuckets` | Policy | **Neutraliza lo existente.** Si el bucket ya tiene una política pública, solo pueden usarla los servicios de AWS y las identidades de tu propia cuenta. Bloquea el acceso anónimo y el de otras cuentas. | Una política pública creada antes de activar el bloqueo deja de funcionar para usuarios anónimos. |


In [20]:
bloqueo = s3.get_public_access_block(Bucket=BUCKET)["PublicAccessBlockConfiguration"]
for opcion, valor in bloqueo.items():
    print(f"{'✅' if valor else '❌'} {opcion}: {valor}")

assert all(bloqueo.values()), "Hay acceso público habilitado: revisa la configuración del bucket"

✅ BlockPublicAcls: True
✅ IgnorePublicAcls: True
✅ BlockPublicPolicy: True
✅ RestrictPublicBuckets: True


## Ejercicio 3 · Subir un archivo puntual a la zona `raw/`

Equivale a `aws s3 cp data/clientes.csv s3://$BUCKET/raw/clientes/`.

`upload_file` divide automáticamente los archivos grandes en partes (multipart upload).

In [21]:
clientes = STAGING / "2026-10-27" / "clientes.csv"
key = f"raw/clientes/{clientes.name}"

s3.upload_file(str(clientes), BUCKET, key)
print(f"subido: s3://{BUCKET}/{key}")

subido: s3://de-tecsup-marcell/raw/clientes/clientes.csv


## Ejercicio 4 · Sincronizar una carpeta

`aws s3 sync` compara y sube **solo lo nuevo o modificado**; `cp` copia siempre. boto3 no trae `sync`, así que lo implementamos con la misma regla que la CLI: subir si el objeto no existe, si cambió el tamaño o si el archivo local es más reciente.

Usamos los reportes diarios del ETL, que ya son agregados de negocio: van a la zona `curated/`.

In [24]:
def objetos_remotos(prefijo: str) -> dict:
    """key -> (tamaño, fecha de modificación) de todo lo que hay bajo un prefijo."""
    remotos = {}
    for pagina in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=prefijo):
        for obj in pagina.get("Contents", []):
            remotos[obj["Key"]] = (obj["Size"], obj["LastModified"])
    return remotos


def sync(carpeta_local: Path, prefijo: str) -> None:
    remotos = objetos_remotos(prefijo)
    subidos = 0
    for archivo in sorted(p for p in carpeta_local.rglob("*") if p.is_file()):
        key = prefijo + archivo.relative_to(carpeta_local).as_posix()
        info = archivo.stat()
        local_mtime = datetime.fromtimestamp(info.st_mtime, tz=timezone.utc)
        remoto = remotos.get(key)
        if remoto is None or remoto[0] != info.st_size or local_mtime > remoto[1]:
            s3.upload_file(str(archivo), BUCKET, key)
            print("upload:", key)
            subidos += 1
    print(f"{subidos} archivo(s) subido(s)")


sync(REPORTES, "curated/reportes/")

upload: curated/reportes/reporte_2026-10-20.csv
upload: curated/reportes/reporte_2026-10-21.csv
upload: curated/reportes/reporte_2026-10-22.csv
upload: curated/reportes/reporte_2026-10-23.csv
upload: curated/reportes/reporte_2026-10-24.csv
upload: curated/reportes/reporte_2026-10-25.csv
upload: curated/reportes/reporte_2026-10-26.csv
upload: curated/reportes/reporte_2026-10-27.csv
8 archivo(s) subido(s)


Ejecuta `sync` otra vez: como nada cambió, no debería subir ningún archivo.

In [25]:
sync(REPORTES, "curated/reportes/")

0 archivo(s) subido(s)


## Ejercicio 5 · Listar con tamaños legibles

Equivale a `aws s3 ls s3://$BUCKET/ --recursive --human-readable`. Recuerda: las carpetas no existen, son prefijos de la key.

In [26]:
def tamano_legible(n: float) -> str:
    for unidad in ["B", "KiB", "MiB", "GiB", "TiB"]:
        if n < 1024:
            return f"{n:.1f} {unidad}"
        n /= 1024
    return f"{n:.1f} PiB"


def listar(prefijo: str = "") -> None:
    total = 0
    for key, (tamano, fecha) in sorted(objetos_remotos(prefijo).items()):
        print(f"{fecha:%Y-%m-%d %H:%M:%S} {tamano_legible(tamano):>10}  {key}")
        total += tamano
    print(f"\nTotal: {tamano_legible(total)}")


listar()

2026-11-13 02:53:32    267.0 B  curated/reportes/reporte_2026-10-20.csv
2026-11-13 02:53:32    266.0 B  curated/reportes/reporte_2026-10-21.csv
2026-11-13 02:53:32    269.0 B  curated/reportes/reporte_2026-10-22.csv
2026-11-13 02:53:33    271.0 B  curated/reportes/reporte_2026-10-23.csv
2026-11-13 02:53:33    267.0 B  curated/reportes/reporte_2026-10-24.csv
2026-11-13 02:53:33    267.0 B  curated/reportes/reporte_2026-10-25.csv
2026-11-13 02:53:33    266.0 B  curated/reportes/reporte_2026-10-26.csv
2026-11-13 02:53:34    269.0 B  curated/reportes/reporte_2026-10-27.csv
2026-11-13 02:43:25    5.2 KiB  raw/clientes/clientes.csv

Total: 7.3 KiB


## Ejercicio 6 · Subir ventas particionadas en Parquet (diapositiva 19)

Por cada día de `staging/`:

1. Lee `ventas.csv`.
2. Lo convierte a **Parquet**, columnar y comprimido con Snappy.
3. Arma la key estilo Hive: `processed/ventas/year=YYYY/month=MM/day=DD/ventas.parquet`.
4. Lo sube con `upload_file`.

Los Parquet se escriben en una carpeta temporal para no ensuciar `staging/`.

In [27]:
with tempfile.TemporaryDirectory() as tmp:
    for archivo_csv in sorted(STAGING.glob("*/ventas.csv")):
        fecha = archivo_csv.parent.name          # "2026-10-22"
        y, m, d = fecha.split("-")

        local = Path(tmp) / f"ventas_{fecha}.parquet"
        pd.read_csv(archivo_csv).to_parquet(local, index=False)

        key = f"processed/ventas/year={y}/month={m}/day={d}/ventas.parquet"
        s3.upload_file(str(local), BUCKET, key)
        print("subido:", key)

subido: processed/ventas/year=2026/month=10/day=20/ventas.parquet
subido: processed/ventas/year=2026/month=10/day=21/ventas.parquet
subido: processed/ventas/year=2026/month=10/day=22/ventas.parquet
subido: processed/ventas/year=2026/month=10/day=23/ventas.parquet
subido: processed/ventas/year=2026/month=10/day=24/ventas.parquet
subido: processed/ventas/year=2026/month=10/day=25/ventas.parquet
subido: processed/ventas/year=2026/month=10/day=26/ventas.parquet
subido: processed/ventas/year=2026/month=10/day=27/ventas.parquet


In [28]:
listar("processed/ventas/")

2026-11-13 03:03:16    7.1 KiB  processed/ventas/year=2026/month=10/day=20/ventas.parquet
2026-11-13 03:03:16    7.1 KiB  processed/ventas/year=2026/month=10/day=21/ventas.parquet
2026-11-13 03:03:17    7.4 KiB  processed/ventas/year=2026/month=10/day=22/ventas.parquet
2026-11-13 03:03:17    7.4 KiB  processed/ventas/year=2026/month=10/day=23/ventas.parquet
2026-11-13 03:03:17    7.1 KiB  processed/ventas/year=2026/month=10/day=24/ventas.parquet
2026-11-13 03:03:18    7.2 KiB  processed/ventas/year=2026/month=10/day=25/ventas.parquet
2026-11-13 03:03:18    7.0 KiB  processed/ventas/year=2026/month=10/day=26/ventas.parquet
2026-11-13 03:03:18    7.2 KiB  processed/ventas/year=2026/month=10/day=27/ventas.parquet

Total: 57.6 KiB


## Ejercicio 7 · Leer de vuelta desde S3

Descargamos un día a memoria y luego armamos el mes completo, convirtiendo las carpetas `year=/month=/day=` en columnas

In [29]:
key = "processed/ventas/year=2026/month=10/day=22/ventas.parquet"
cuerpo = s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()
df_dia = pd.read_parquet(io.BytesIO(cuerpo))
print(f"{len(df_dia)} filas en {key}")
df_dia.head()

100 filas en processed/ventas/year=2026/month=10/day=22/ventas.parquet


,producto_id,nombre,categoria,precio_unitario,cantidad,pedido_id,fecha,cliente_id,canal,monto
0,3,Teclado,Tecnologia,120.0,4,ERP-20261022-001,2026-10-22,2,tienda,480.0
1,8,Lapicero,Libreria,2.5,2,ERP-20261022-001,2026-10-22,2,tienda,5.0
2,7,Cuaderno,Libreria,12.5,4,ERP-20261022-001,2026-10-22,2,tienda,50.0
3,5,Silla ergonomica,Muebles,690.0,4,ERP-20261022-002,2026-10-22,77,tienda,2760.0
4,1,Laptop,Tecnologia,3200.0,2,ERP-20261022-002,2026-10-22,77,tienda,6400.0


In [30]:
partes = []
for key in sorted(objetos_remotos("processed/ventas/")):
    particion = dict(p.split("=") for p in key.split("/") if "=" in p)   # {'year': '2026', ...}
    cuerpo = s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()
    partes.append(pd.read_parquet(io.BytesIO(cuerpo)).assign(**particion))

ventas = pd.concat(partes, ignore_index=True)
ventas.groupby(["year", "month", "day"])["monto"].agg(["count", "sum"])

count       sum
year month day                 
2026 10    20      75   95515.0
           21      73  139457.5
           22     100  148072.5
           23     107  165202.5
           24      78  115862.5
           25      85  146990.0
           26      69   77242.5
           27      91   91972.5

## Ejercicio 8 · Limpieza (opcional)

Al terminar el curso, cambia `BORRAR` a `True` para vaciar y eliminar el bucket (equivale a `aws s3 rb s3://$BUCKET --force`).

> **Orden de limpieza:** de atrás hacia adelante, **04 → 03 → 02 → 01**. El bucket (notebook 01) se borra al final, porque Athena necesita su carpeta `athena-results/` incluso para ejecutar un `DROP`.

In [ ]:
BORRAR = True

if BORRAR:
    bucket = session.resource("s3").Bucket(BUCKET)
    bucket.objects.all().delete()
    bucket.object_versions.all().delete()   # por si activaste el versionado
    bucket.delete()
    print(f"Bucket s3://{BUCKET} eliminado")